# AI Data Platform / Lakehouse Demo

## Project Question

**Can a modern healthcare/public-sector lakehouse improve scalable AI analytics and forecasting workflows?**

This notebook demonstrates:

1. synthetic public-sector/healthcare style event generation  
2. Bronze / Silver / Gold lakehouse layers  
3. data quality checks  
4. feature store creation  
5. ML forecasting model  
6. governance and dashboard outputs  


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
print("ROOT:", ROOT)

ROOT: /Users/yuzhang/projects/Machine_learning/08_ai_data_platform_lakehouse


## 1. Generate synthetic public-sector / healthcare events

In [2]:
from src.data_generator import generate_synthetic_public_health_events

df = generate_synthetic_public_health_events(
    n_records=10000,
    output_path=ROOT / "data/raw/public_health_events.csv"
)

display(df.head())
print(df.shape)

,event_id,event_month,county,program,site_id,age_band,risk_score,service_count,prior_utilization,social_need_index,target_next_month_demand
0,EVT_00000000,2025-02-01,Los Angeles,Clinic_Access,SITE_014,35-54,0.097401,4,2,0.838718,38.274974
1,EVT_00000001,2026-02-01,Los Angeles,Enrollment_Support,SITE_021,55+,0.464585,3,3,0.437515,34.853748
2,EVT_00000002,2025-12-01,San Diego,Clinic_Access,SITE_052,0-17,0.329381,3,2,0.443887,32.344823
3,EVT_00000003,2025-08-01,Riverside,Enrollment_Support,SITE_007,18-34,0.695442,4,2,0.569549,39.198067
4,EVT_00000004,2025-08-01,Los Angeles,Enrollment_Support,SITE_007,35-54,0.318247,4,4,0.214808,36.538014


(10000, 11)


## 2. Run Bronze/Silver/Gold pipeline

In [3]:
from src.lakehouse_pipeline import bronze_ingest, silver_clean, gold_certified_tables

bronze = bronze_ingest(
    raw_path=ROOT / "data/raw/public_health_events.csv",
    bronze_path=ROOT / "data/bronze/events_bronze.parquet"
)

silver = silver_clean(
    bronze_path=ROOT / "data/bronze/events_bronze.parquet",
    silver_path=ROOT / "data/silver/events_silver.parquet"
)

gold = gold_certified_tables(
    silver_path=ROOT / "data/silver/events_silver.parquet",
    gold_path=ROOT / "data/gold/gold_monthly_program_demand.parquet"
)

display(gold.head())
print(gold.shape)

Saved Gold parquet to: /Users/yuzhang/projects/Machine_learning/08_ai_data_platform_lakehouse/data/gold/gold_monthly_program_demand.parquet
Saved Gold CSV to: /Users/yuzhang/projects/Machine_learning/08_ai_data_platform_lakehouse/outputs/tables/gold_monthly_program_demand.csv


,event_month,county,program,total_services,avg_risk_score,avg_social_need,total_prior_utilization,target_next_month_demand,record_count
0,2025-01-01,Los Angeles,Clinic_Access,99,0.323411,0.406104,54,1009.880746,33
1,2025-01-01,Los Angeles,Enrollment_Support,81,0.257536,0.459913,35,671.051780,20
2,2025-01-01,Los Angeles,Preventive_Care,102,0.325550,0.457916,54,947.302179,30
3,2025-01-01,Los Angeles,Telehealth,62,0.361862,0.502719,29,611.377561,22
4,2025-01-01,Orange,Clinic_Access,88,0.253666,0.407086,57,859.129357,27


(360, 9)


## 3. Data quality checks

In [4]:
from src.quality import run_quality_checks

quality = run_quality_checks(
    silver_path=ROOT / "data/silver/events_silver.parquet",
    output_path=ROOT / "outputs/tables/data_quality_report.csv"
)

display(quality)

,check_name,passed,value
0,row_count_positive,True,10000
1,event_id_unique,True,10000
2,county_not_null,True,0
3,program_not_null,True,0
4,site_id_not_null,True,0
5,event_month_not_null,True,0
6,risk_score_between_0_and_1,True,0.002 - 0.922
7,social_need_index_between_0_and_1,True,0.000 - 1.000


## 4. Build feature store

In [5]:
from src.feature_store import build_feature_store

features = build_feature_store(
    gold_path=ROOT / "data/gold/gold_monthly_program_demand.parquet",
    feature_path=ROOT / "data/feature_store/monthly_program_features.parquet"
)

display(features.head())
print(features.shape)

OSError: Cannot save file into a non-existent directory: 'outputs/tables'

## 5. Train forecasting model

In [ ]:
from ml.train_forecast_model import train_forecasting_model

metrics = train_forecasting_model(
    feature_path=ROOT / "data/feature_store/monthly_program_features.parquet",
    model_path=ROOT / "outputs/models/demand_forecast_model.joblib",
    metrics_path=ROOT / "outputs/tables/forecast_model_metrics.json",
    predictions_path=ROOT / "outputs/tables/forecast_predictions.csv"
)

metrics

## 6. Generate visual outputs

In [ ]:
from src.visualization import generate_figures

figures = generate_figures(
    gold_csv=ROOT / "outputs/tables/gold_monthly_program_demand.csv",
    predictions_csv=ROOT / "outputs/tables/forecast_predictions.csv",
    metrics_json=ROOT / "outputs/tables/forecast_model_metrics.json",
    quality_csv=ROOT / "outputs/tables/data_quality_report.csv",
    output_dir=ROOT / "outputs/figures"
)

figures

## Final Interpretation

This project demonstrates a modern AI data platform pattern: raw events are transformed into certified Gold datasets and reusable feature-store tables, then used for scalable forecasting.

It is portfolio-safe because the dataset is synthetic, but the architecture mirrors real healthcare/public-sector lakehouse workflows.
